In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_dev")
dbutils.widgets.text("source", "bronze")
dbutils.widgets.text("sink", "silver")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
schema_source = dbutils.widgets.get("source")
schema_sink = dbutils.widgets.get("sink")

In [0]:
df_producto = spark.table(f"{catalogo}.{schema_source}.productos")
df_venta = spark.table(f"{catalogo}.{schema_source}.ventas")

In [0]:
df_producto_select = df_producto.select("ID_ITEM", "PRODUCT", "CATEGORY")
display(df_producto_select)

In [0]:
df_venta_valid = df_venta.filter("STATUS='VALID'")
df_venta_select = df_venta_valid.select("ID_ORDER", "STORE", "REG", "DATE", "ID_ITEM", "QTY", "PRICE", "DISCOUNT", "ID_SELLER")
display(df_venta_select)


In [0]:
df_producto_alias = df_producto_select.alias("producto")
df_venta_alias = df_venta_select.alias("venta")

df_join = F.broadcast(df_producto_alias).join(
    df_venta_alias,
    F.col("producto.ID_ITEM") == F.col("venta.ID_ITEM"),
    "inner"
).select(
    F.col("producto.ID_ITEM").alias("ID_ARTICULO"),
    F.col("producto.PRODUCT").alias("PRODUCTO"),
    F.col("producto.CATEGORY").alias("CATEGORIA"),
    F.col("venta.ID_ORDER").alias("ID_PEDIDO"),
    F.col("venta.STORE").alias("TIENDA"),
    F.col("venta.REG").alias("NUM_LINEA"),
    F.col("venta.DATE").alias("FECHA"),
    F.col("venta.QTY").alias("CANTIDAD"),
    F.col("venta.PRICE").alias("PRECIO"),
    F.col("venta.DISCOUNT").alias("DESCUENTO"),
    F.col("venta.ID_SELLER").alias("ID_VENDEDOR")
)

display(df_join)

In [0]:
def estado(descuento):
    if descuento == 0:
        return "Precio Normal"
    elif descuento == 100:
        return "Regalo"
    elif descuento > 75 and descuento < 100:
        return "bono empresarial"
    elif descuento > 1 and descuento <= 74:
        return "promoción"
    else:
        return None

precio_descuento_udf = F.udf(estado, StringType())

df_join_etiqueta = df_join.withColumn("ETIQUETA_DESCUENTO", precio_descuento_udf(F.col("DESCUENTO")))
display(df_join_etiqueta)

df_join_etiqueta.coalesce(4).write.mode("overwrite").option("overwriteSchema", "true").insertInto(f"{catalogo}.{schema_sink}.ventas_test")

In [0]:
target_table = f"{catalogo}.{schema_sink}.ventas_productos_categorias"

(df_join
    .withColumn(
        "ETIQUETA_DESCUENTO",
        F.when(F.col("DESCUENTO") == 0, F.lit("Precio Normal"))
         .when(F.col("DESCUENTO") == 100, F.lit("Regalo"))
         .when((F.col("DESCUENTO") > 75) & (F.col("DESCUENTO") < 100), F.lit("bono empresarial"))
         .when((F.col("DESCUENTO") > 1) & (F.col("DESCUENTO") <= 74), F.lit("promoción"))
    )
    .select(
        F.col("ID_ARTICULO").cast("string").alias("ID_ARTICULO"),
        F.col("PRODUCTO").cast("string").alias("PRODUCTO"),
        F.col("CATEGORIA").cast("string").alias("CATEGORIA"),
        F.col("ID_PEDIDO").cast("string").alias("ID_PEDIDO"),
        F.col("TIENDA").cast("string").alias("TIENDA"),
        F.col("NUM_LINEA").cast("string").alias("NUM_LINEA"),
        F.col("FECHA").cast("date").alias("FECHA"),
        F.col("CANTIDAD").cast("int").alias("CANTIDAD"),
        F.col("PRECIO").cast("double").alias("PRECIO"),
        F.col("DESCUENTO").cast("int").alias("DESCUENTO"),
        F.col("ID_VENDEDOR").cast("string").alias("ID_VENDEDOR"),
        F.col("ETIQUETA_DESCUENTO").cast("string").alias("ETIQUETA_DESCUENTO")
    )
    .coalesce(4)
    .write
    .mode("overwrite")
    .insertInto(target_table)
)